# Integração entre Dados das API's do IBGE e BC

Este notebook integrará os dados recebidos pelas API's do IBGE e do BC e irá uni-los em uma só tabela

## Imports

In [29]:
import json
import pandas as pd
import requests as rqs
from pathlib import Path
import altair as alt
import geobr
import duckdb
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
from IPython.display import display
from ipywidgets import Dropdown, Layout, interactive_output

## Reunir os arquivos da API do BC

Os dados da API do BC estão na pasta dados > bcb > cache em formato tipo .json

In [30]:
dados_da_api_bc = []

# Pesquisa todos os arquivos .json em dados > bcb > cache
for arquivo in Path("dados/bcb/cache").glob("*.json"):
    with open(arquivo, encoding="utf-8") as f:
        cache = json.load(f)
    dados_da_api_bc.append(pd.DataFrame(cache["dados"]["value"]))

sicor = pd.concat(dados_da_api_bc, ignore_index=True)

## Rodar o código da API_IBGE

Rodando o código com o %run e colocando o %%capture para não dar print em nada na tela, ou seja, apenas rodar e carregar todas as tabelas do API_IBGE

In [31]:
%%capture
%run API_IBGE.ipynb

## Padronizando os dados da API_BC

Padronizando os dados da API_BC para igualar os valores dos dados aos da API_IBGE e tratar a tabela

In [32]:
# Renomeando as colunas com o mesmo nome das colunas da API_IBGE
sicor = sicor.rename(columns = {'nomeProduto':'cultura', 'AnoEmissao':'ano', 'MesEmissao':'mes', 'Municipio':'municipio' })

# Padronizando o nome da cultura
sicor['cultura'] = sicor['cultura'].replace({
    '"SOJA"':'Soja (em grão)',
    '"CANA-DE-ALÚCAR"' :'Cana-de-açúcar',
    '"MILHO"':'Milho (em grão)'
})

# Deixando o municipio em letra minúscula
sicor['municipio'] = sicor['municipio'].str.title()
sicor
# Renomenado as colunas relacionadas com Crédito rural
sicor = sicor.rename(columns={
    'cdFonteRecurso': 'fonte de recurso',
    'cdPrograma': 'programa de financiamento',
    'cdSubPrograma': 'subprograma',
    'cdModalidade': 'modalidade',
    'Atividade': 'atividade',
    'cdTipoSeguro': 'tipo de seguro',
    'VlCusteio': 'valor de custeio',
    'AreaCusteio': 'area de custeio',
})

#cria a tabela com os dados úteis do BC
tabela_bc = sicor.drop(columns=['codCadMu', 'cdProduto', 'codIbge','cdEstado'])

## Padronizando os dados da API_IBGE

A padronização será adicionar todos os dados municipais em uma única tabela

In [33]:
#Unindo todas as tabelas com base nas colunas iguais
tabela_ibge_municipal = df_area_municipal.merge(df_prod_municipal, on=['cultura','codigo_municipio','ano','municipio','codigo_cultura'])
tabela_ibge_municipal = tabela_ibge_municipal.merge(df_rendimento_municipal, on=['cultura','codigo_municipio','ano','municipio','codigo_cultura'])

#Renomendo as colunas e retirando as colunas redundantes
tabela_ibge_municipal = tabela_ibge_municipal.drop(columns=['variavel_x','unidade_x','variavel_y','unidade_y', 'variavel','unidade'])
tabela_ibge_municipal = tabela_ibge_municipal.rename(columns={
    'valor_x':'area plantada (hectares)',
    'valor_y':'quantidade produzida (toneladas)',
    'valor':'rendimento medio (kg por hectare)',
    'codigo_cultura':'codigo cultura',
    'codigo_municipio':'codigo municipio'
})

# Adicionando a sigla do estado
tabela_ibge_municipal['estado'] = tabela_ibge_municipal['municipio'].str.split('-').str[1].str.strip()

# Padronizando o municipio para ficar idêntico ao da API_BC
tabela_ibge_municipal['municipio'] = tabela_ibge_municipal['municipio'].str.split("-").str[0].str.strip().str.title().astype(str)

# Ordenar
nova_ordem = [
    'ano',
    'estado',
    'municipio',
    'codigo municipio',
    'cultura',
    'codigo cultura',
    'area plantada (hectares)',
    'quantidade produzida (toneladas)',
    'rendimento medio (kg por hectare)'
]
tabela_ibge_municipal = tabela_ibge_municipal[nova_ordem]
tabela_ibge_municipal

,ano,estado,municipio,codigo municipio,cultura,codigo cultura,area plantada (hectares),quantidade produzida (toneladas),rendimento medio (kg por hectare)
0,2016,GO,Abadia De Goiás,5200050,Cana-de-açúcar,2696,0.0,0.0,0.0
1,2017,GO,Abadia De Goiás,5200050,Cana-de-açúcar,2696,5.0,300.0,60000.0
2,2018,GO,Abadia De Goiás,5200050,Cana-de-açúcar,2696,4.0,238.0,59500.0
3,2019,GO,Abadia De Goiás,5200050,Cana-de-açúcar,2696,15.0,1208.0,80533.0
4,2020,GO,Abadia De Goiás,5200050,Cana-de-açúcar,2696,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...
164758,2021,SP,Óleo,3533809,Soja (em grão),2713,5890.0,24792.0,4209.0
164759,2022,SP,Óleo,3533809,Soja (em grão),2713,5890.0,24765.0,4205.0
164760,2023,SP,Óleo,3533809,Soja (em grão),2713,5143.0,20402.0,3967.0
164761,2024,SP,Óleo,3533809,Soja (em grão),2713,5143.0,10243.0,1992.0


## Dicionário para padronização da 2° tabela (a seguir)

In [34]:
programas = {
    "0001": "PRONAF",
    "0050": "PRONAMP",
    "0999": "Sem vínculo a programa específico",
    "0070": "Funcafé",
    "0151": "Procap-Agro",
    "0152": "Proirriga (antigo Moderinfra)",
    "0153": "Moderagro",
    "0154": "Moderfrota",
    "0155": "Prodecoop",
    "0162": "Inovagro",
    "0163": "PCA (armazéns)",
    "0201": "Crédito Fundiário (FTRA)",
    "0222": "Renovagro",
    "0240": "ANF (atividade não financiada, Proagro)",
}

fontes = {
    "0100": "TESOURO NACIONAL - DIRECIONADA/CONTROLADA",
    "0201": "OBRIGATÓRIOS - MCR 6.2 - DIRECIONADA/CONTROLADA",
    "0202": "EXIGIBILIDADE ADICIONAL DOS RECURSOS À VISTA",
    "0222": "Exigibilidade Adicional dos Recursos à Vista - Resolução 5030 ENCERRADO",
    "0225": "Exigibilidade Adicional dos Recursos à Vista - Resolução 5087 - ENCERRADO",
    "0226": "Exigibilidade Adicional dos Recursos à Vista - Resolução 5157",
    "0250": "FACULDADE DE APLICAÇÃO - COMPULSÓRIO",
    "0260": "COMPULSÓRIO SOBRE RECURSOS À VISTA - REFORÇO DO INVESTIMENTO (CIRC 3.745)",
    "0300": "POUPANÇA RURAL - EQUALIZADA - DIRECIONADA/CONTROLADA",
    "0301": "POUPANÇA RURAL - CONTROLADOS - CONDIÇÕES MCR 6.2",
    "0302": "POUPANÇA RURAL - CONTROLADOS - FATOR DE PONDERAÇÃO",
    "0303": "POUPANÇA RURAL - DIRECIONADA/NÃO CONTROLADA",
    "0304": "EXIGIBILIDADE ADICIONAL DA POUPANÇA RURAL",
    "0402": "RECURSOS LIVRES - LIVRE/NÃO CONTROLADA",
    "0403": "RECURSOS LIVRES - EQUALIZADA - LIVRE/CONTROLADA",
    "0405": "FUNDO DE COMMODITIES",
    "0430": "LETRA DE CRÉDITO DO AGRONEGÓCIO (LCA) - DIRECIONADA/NÃO CONTROLADA",
    "0431": "LETRA DE CRÉDITO DO AGRONEGÓCIO (LCA) - EQUALIZADA - DIRECIONADA/CONTROLADA",
    "0440": "LETRA DE CRÉDITO DO AGRONEGÓCIO (LCA) - TAXA FAVORECIDA",
    "0450": "INSTR HIBRIDO CAPITAL DÍVIDA-IHCD (Lei 12.793/2013 - Art. 6º) - EQUALIZÁVEL",
    "0451": "INSTR HIBRIDO CAPITAL DÍVIDA-IHCD (Lei 12.793/2013 - Art. 6º) - DIRECIONADA",
    "0501": "FUNDO CONSTITUCIONAL DE FINANCIAMENTO DO NORTE (FNO) - DIRECIONADA/CONTROLADA",
    "0502": "FUNDO CONSTITUCIONAL DE FINANCIAMENTO DO NORDESTE (FNE) - DIRECIONADA/CONTROLADA",
    "0503": "FUNDO CONSTITUCIONAL DE FINANCIAMENTO DO CENTRO-OESTE (FCO) - DIRECIONADA/CONTROLADA",
    "0505": "BNDES/FINAME - EQUALIZADA - DIRECIONADA/CONTROLADA",
    "0506": "BNDES - LIVRE/NÃO CONTROLADA",
    "0507": "INCRA - DIRECIONADA/CONTROLADA",
    "0520": "FUNDO DE TERRAS E DA REFORMA AGRÁRIA - DIRECIONADA/CONTROLADA",
    "0600": "GOVERNOS E FUNDOS ESTADUAIS OU MUNICIPAIS - DIRECIONADA/CONTROLADA",
    "0650": "FAT - FUNDO DE AMPARO AO TRABALHADOR - DIRECIONADA/CONTROLADA",
    "0680": "PIS/PASEP",
    "0800": "FUNCAFE - FUNDO DE DEFESA DA ECONOMIA CAFEEIRA - DIRECIONADA/CONTROLADA",
    "0850": "CAPTAÇÃO EXTERNA - LIVRE/NÃO CONTROLADA",
    "0900": "ATIVIDADE NÃO FINANCIADA ENQUADRADA NO PROAGRO (MCR 16.8)",
    "0910": "FUNDO NACIONAL SOBRE MUDANÇA (FUNDO CLIMA) - RES CMN 5.130/2024",
    "0911": "FUNDO DE DESENVOLVIMENTO CIENTÍFICO E TECNOLÓGICO (FNDCT) - MP 1374 DIRECIONADA/CONTROLADA",
    "0990": "OUTRAS FONTES DE RECURSOS NÃO ESPECIFICADAS",
}

tipo_seguro = {
    0: "Não se aplica",
    1: "Proagro tradicional",
    2: "Proagro mais",
    3: "Outro seguro",
    9: "Sem adesão a seguro",
}

meses = {
    "01": "Janeiro",
    "02": "Fevereiro",
    "03": "Março",
    "04": "Abril",
    "05": "Maio",
    "06": "Junho",
    "07": "Julho",
    "08": "Agosto",
    "09": "Setembro",
    "10": "Outubro",
    "11": "Novembro",
    "12": "Dezembro"
}


## Criação de duas tabelas com os dados do BC

1° Tabela: tabela agrupada com o valor do crédito rural por municipio, cultura e ano

2° Tabela: tabela padronizada com os nomes de cada coluna corretamente

In [35]:
# 1° Tabela: tabela do crédito rural
colunas_a_serem_somadas = ['valor de custeio', 'area de custeio']
tabela_credito_rural = sicor.groupby(['municipio','cultura','ano'])[colunas_a_serem_somadas].sum().reset_index()
# Transformando o ano em int
tabela_credito_rural['ano'] = tabela_credito_rural['ano'].astype(int)

# 2° Tabela: tabela com os nomes padronizados com o dicionário de cima
tabela_padronizada = sicor.copy()
tabela_padronizada['programa de financiamento'] = sicor['programa de financiamento'].replace(programas)
tabela_padronizada['fonte de recurso'] = sicor['fonte de recurso'].replace(fontes)
tabela_padronizada['tipo de seguro'] = sicor['tipo de seguro'].replace(tipo_seguro)
# ANALISAR SE PRECISA PADRONIZAR O MÊS
# tabela_padronizada['mes'] = sicor['mes'].replace(meses)
tabela_padronizada['mes'] = sicor['mes'].astype(int)

tabela_padronizada


,municipio,cultura,mes,ano,programa de financiamento,subprograma,fonte de recurso,tipo de seguro,cdEstado,valor de custeio,cdProduto,codCadMu,atividade,modalidade,codIbge,area de custeio
0,Jesuítas,Soja (em grão),7,2021,Sem vínculo a programa específico,0,POUPANÇA RURAL - DIRECIONADA/NÃO CONTROLADA,9,19,120515.25,6720,40785,1,01,4112751,40.29
1,Flor Da Serra Do Sul,Soja (em grão),7,2021,PRONAF,1,POUPANÇA RURAL - EQUALIZADA - DIRECIONADA/CONT...,3,19,884242.51,6720,49982,1,01,4107850,264.96
2,Dianópolis,Soja (em grão),7,2021,Sem vínculo a programa específico,0,LETRA DE CRÉDITO DO AGRONEGÓCIO (LCA) - DIRECI...,3,28,354260.00,6720,12036,1,01,1707009,100.00
3,Nova Araçá,Soja (em grão),7,2021,PRONAMP,0,BNDES/FINAME - EQUALIZADA - DIRECIONADA/CONTRO...,1,24,44311.34,6720,34832,1,01,4312807,12.00
4,Iporã,Soja (em grão),7,2021,PRONAMP,0,BNDES/FINAME - EQUALIZADA - DIRECIONADA/CONTRO...,3,19,129270.72,6720,30953,1,01,4110607,42.50
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
831985,Caseara,Soja (em grão),1,2018,Sem vínculo a programa específico,0,POUPANÇA RURAL - DIRECIONADA/NÃO CONTROLADA,9,28,1150000.00,6720,43751,1,01,1703909,564.89
831986,Presidente Bernardes,Soja (em grão),1,2018,PRONAF,1,POUPANÇA RURAL - EQUALIZADA - DIRECIONADA/CONT...,2,27,27267.81,6720,40163,1,01,3541208,16.94
831987,Nobres,Soja (em grão),1,2018,Sem vínculo a programa específico,0,OBRIGATÓRIOS - MCR 6.2 - DIRECIONADA/CONTROLADA,3,14,2606343.93,6720,12988,1,01,5105903,1278.70
831988,Águas Frias,Soja (em grão),1,2018,PRONAF,1,POUPANÇA RURAL - EQUALIZADA - DIRECIONADA/CONT...,2,25,5892.00,6720,49494,1,01,4200556,4.00


## Unir as tabelas

Irá unir-se as tabelas: tabela_ibge_municipal com tabela_credito_rural. Entretanto, há alguns valores que estão apenas em uma das duas tabelas, logo esses valores possuirão NaN e serão removidos.

In [36]:
tabela_unida = tabela_ibge_municipal.merge(tabela_credito_rural, on=['municipio', 'ano','cultura'])
tabela_unida = tabela_unida.dropna()
tabela_unida

,ano,estado,municipio,codigo municipio,cultura,codigo cultura,area plantada (hectares),quantidade produzida (toneladas),rendimento medio (kg por hectare),valor de custeio,area de custeio
0,2021,GO,Abadia De Goiás,5200050,Milho (em grão),2711,200.0,1360.0,6800.0,1220142.65,463.97
1,2023,GO,Abadia De Goiás,5200050,Milho (em grão),2711,300.0,1332.0,4440.0,1365264.35,394.46
2,2016,GO,Abadia De Goiás,5200050,Soja (em grão),2713,0.0,0.0,0.0,293962.23,160.55
3,2017,GO,Abadia De Goiás,5200050,Soja (em grão),2713,217.0,720.0,3318.0,399997.72,209.39
4,2018,GO,Abadia De Goiás,5200050,Soja (em grão),2713,197.0,621.0,3152.0,405726.63,209.39
...,...,...,...,...,...,...,...,...,...,...,...
49099,2021,SP,Óleo,3533809,Soja (em grão),2713,5890.0,24792.0,4209.0,3529324.92,1043.23
49100,2022,SP,Óleo,3533809,Soja (em grão),2713,5890.0,24765.0,4205.0,7584973.10,1292.92
49101,2023,SP,Óleo,3533809,Soja (em grão),2713,5143.0,20402.0,3967.0,8117759.43,1607.08
49102,2024,SP,Óleo,3533809,Soja (em grão),2713,5143.0,10243.0,1992.0,6289436.44,1176.02


## Gráficos

### Gráfico 1) Como evoluíram o valor contratado e o número de contratos de custeio por cultura na década?

In [37]:
tabela_valor_contratado = sicor.groupby(['cultura', 'ano','mes'])['valor de custeio'].sum().reset_index()

# Coluna com a data formatada adequadamente
tabela_valor_contratado['data'] = pd.to_datetime({
    'year': tabela_valor_contratado['ano'],
    'month': tabela_valor_contratado['mes'],
    # o dia é padronizado como o 1° dia de cada mês, porque a função exige que se passe um dia
    'day': 1
})

cores = alt.Scale(
    domain=['Soja (em grão)', 'Milho (em grão)', 'Cana-de-açúcar'],
    range=['red','orange','green']
)

grafico_valor_contratado = alt.Chart(tabela_valor_contratado).mark_line().encode(
    x=alt.X('yearmonth(data):T', 
        title='Mês / Ano',
        axis=alt.Axis(format="%m/%Y", tickCount={"interval": "month", "step": 6})
    ),
    y=alt.Y('valor de custeio:Q',
        title='Valor de Custeio'
    ),
    color=alt.Color('cultura:N', scale=cores, title= 'Culturas')
).properties(
    title='Evolução do Valor de Custeio por Cultura',
    width=700,
    height=400
)
grafico_valor_contratado

alt.Chart(...)

In [38]:
tabela_numero_contratos = sicor.groupby(['cultura','ano','mes']).size().reset_index(name='quantidade')

# Coluna com a data formatada adequadamente
tabela_numero_contratos['data'] = pd.to_datetime({
    'year': tabela_numero_contratos['ano'],
    'month': tabela_numero_contratos['mes'],
    # o dia é padronizado como o 1° dia de cada mês, porque a função exige que se passe um dia
    'day': 1
})

cores = alt.Scale(
    domain=['Soja (em grão)', 'Milho (em grão)', 'Cana-de-açúcar'],
    range=['red','orange','green']
)

grafico_numero_contratos = alt.Chart(tabela_numero_contratos).mark_line().encode(
    x=alt.X('yearmonth(data):T', 
        title='Mês / Ano',
        axis=alt.Axis(format="%m/%Y", tickCount={"interval": "month", "step": 6})
    ),
    y=alt.Y('quantidade:Q',
        title='Quantidade de Contratos',
    ),
    color=alt.Color('cultura:N', scale=cores, title= 'Culturas')
).properties(
    title='Evolução da Quantidade de Contratos por Cultura',
    width=700,
    height=400
)

grafico_numero_contratos

alt.Chart(...)

### Gráfico 2) Como o crédito e a área plantada se distribuem entre estados e regiões?

In [39]:
# Comparar crédito nominal e área plantada nos anos em comum de cada cultura.
regioes = {"1": "Norte", "2": "Nordeste", "3": "Sudeste", "4": "Sul", "5": "Centro-Oeste"}
area_g2 = df_area_estadual.rename(columns={"codigo_estado": "uf", "valor": "Área plantada (ha)"})
area_g2 = area_g2.assign(uf=area_g2["uf"].astype(str), ano=area_g2["ano"].astype(int))
credito_g2 = sicor.assign(
    uf=sicor["codIbge"].astype(str).str[:2],
    ano=sicor["ano"].astype(int),
).rename(columns={"valor de custeio": "Crédito de custeio (R$)"})

# Usar apenas os anos que aparecem nas duas fontes para cada cultura.
chave_ano = ["cultura", "ano"]
anos_comuns = area_g2[chave_ano].drop_duplicates().merge(
    credito_g2[chave_ano].drop_duplicates(), on=chave_ano
)
periodos = anos_comuns.groupby("cultura")["ano"].agg(["min", "max"])

indicadores_g2 = ["Crédito de custeio (R$)", "Área plantada (ha)"]
chave_uf = ["cultura", "uf"]
credito_comum = credito_g2.merge(anos_comuns, on=chave_ano)
area_comum = area_g2.merge(anos_comuns, on=chave_ano)

# Somar cada indicador por UF e juntar os resultados, incluindo UFs com dados em uma só fonte.
credito_por_uf = credito_comum.groupby(chave_uf)[indicadores_g2[0]].sum()
area_por_uf = area_comum.groupby(chave_uf)[indicadores_g2[1]].sum()
totais_g2 = pd.concat([credito_por_uf, area_por_uf], axis=1).fillna(0).reset_index()

nomes_uf = area_g2[["uf", "estado"]].drop_duplicates().set_index("uf")["estado"]
totais_g2["estado"] = totais_g2["uf"].map(nomes_uf).fillna("UF não identificada")
totais_g2["regiao"] = totais_g2["uf"].str[0].map(regioes).fillna("Região não identificada")

limite_estados = 8
cores_g2 = ["#2866A6", "#3A851D"]


def grafico_2(cultura):
    dados = totais_g2[totais_g2["cultura"] == cultura]
    por_regiao = dados.groupby("regiao")[indicadores_g2].sum()
    por_estado = dados.groupby("estado")[indicadores_g2].sum()

    # Ordenar pela área plantada e somar o restante em uma única categoria.
    por_estado = por_estado.sort_values("Área plantada (ha)", ascending=False, kind="stable")
    principais = por_estado.head(limite_estados).copy()
    demais = por_estado.iloc[limite_estados:]
    if not demais.empty:
        principais.loc["Outros"] = demais.sum()

    # Dividir cada indicador pela soma do Brasil.
    total_brasil = por_estado.sum().replace(0, float("nan"))
    participacao_regiao = por_regiao / total_brasil
    participacao_estado = principais / total_brasil


    fig, (ax_regiao, ax_estado) = plt.subplots(2, 1, figsize=(13.5, 7), layout="constrained")
    fig.get_layout_engine().set(hspace=0.25)
    participacao_regiao.plot.barh(ax=ax_regiao, color=cores_g2, legend=False, width=0.8)
    participacao_estado.plot.bar(ax=ax_estado, color=cores_g2, legend=False, rot=30, width=0.8)

    ax_regiao.set(title="Região", xlabel="Participação no total nacional", ylabel="")
    ax_regiao.title.set_fontsize(16)
    ax_estado.set(title="Estados", xlabel="", ylabel="Participação no total nacional")
    ax_estado.title.set_fontsize(16)
    ax_regiao.xaxis.set_major_formatter(PercentFormatter(1))
    ax_estado.yaxis.set_major_formatter(PercentFormatter(1))
    ax_regiao.legend(
        loc="lower center", bbox_to_anchor=(0.5, 1.18), ncol=2, frameon=False,
    )

    for eixo, direcao in [(ax_regiao, "x"), (ax_estado, "y")]:
        eixo.set_axisbelow(True)
        eixo.grid(axis=direcao, alpha=0.22)
        eixo.spines[["top", "right", "left"]].set_visible(False)
    for rotulo in ax_estado.get_xticklabels():
        rotulo.set_ha("right")

    primeiro_ano, ultimo_ano = periodos.loc[cultura]
    fig.suptitle(
        f"Distribuição do crédito de custeio e da área plantada\n"
        f"{cultura} · {primeiro_ano} a {ultimo_ano}",
        fontsize=18,
    )
    ax_regiao.tick_params(axis="y", labelsize=14)
    ax_regiao.tick_params(axis="x", labelsize=12)
    ax_estado.tick_params(axis="both", labelsize=12)
    plt.show()
    plt.close(fig)


seletor_g2 = Dropdown(
    options=periodos.index.tolist(), description="Cultura", layout=Layout(width="330px"),
)
saida_g2 = interactive_output(grafico_2, {"cultura": seletor_g2})
display(seletor_g2, saida_g2)

Dropdown(description='Cultura', layout=Layout(width='330px'), options=('Milho (em grão)', 'Soja (em grão)'), v…

Output()

### Gráfico 3) O crédito está mais concentrado em poucos municípios do que a produção?

In [40]:
credito_g3 = sicor[["cultura", "ano", "codIbge", "valor de custeio"]].rename(
    columns={"codIbge": "municipio_id", "valor de custeio": "Crédito de custeio (R$)"}
)
producao_g3 = df_prod_municipal[["cultura", "ano", "codigo_municipio", "valor"]].rename(
    columns={"codigo_municipio": "municipio_id", "valor": "Produção (t)"}
)
credito_g3["municipio_id"] = credito_g3["municipio_id"].astype(str)
producao_g3["municipio_id"] = producao_g3["municipio_id"].astype(str)
credito_g3["ano"] = credito_g3["ano"].astype(int)
producao_g3["ano"] = producao_g3["ano"].astype(int)

# Comparar os dois indicadores nos mesmos anos para cada cultura.
chave_ano_g3 = ["cultura", "ano"]
anos_g3 = credito_g3[chave_ano_g3].drop_duplicates().merge(
    producao_g3[chave_ano_g3].drop_duplicates(), on=chave_ano_g3
)
periodos_g3 = anos_g3.groupby("cultura")["ano"].agg(["min", "max"])

chave_municipio_g3 = ["cultura", "municipio_id"]
coluna_credito = "Crédito de custeio (R$)"
coluna_producao = "Produção (t)"
credito_municipal = (
    credito_g3.merge(anos_g3, on=chave_ano_g3)
    .groupby(chave_municipio_g3)[coluna_credito].sum()
)
producao_municipal = (
    producao_g3.merge(anos_g3, on=chave_ano_g3)
    .groupby(chave_municipio_g3)[coluna_producao].sum()
)
# Incluir municípios com dados em uma só fonte, usando zero no outro indicador.
totais_g3 = pd.concat([credito_municipal, producao_municipal], axis=1).fillna(0).reset_index()


def grafico_3(cultura):
    dados = totais_g3[totais_g3["cultura"] == cultura]
    faixas = [1, 5, 10]
    rotulos = [f"Top {percentual}%" for percentual in faixas] + ["Demais municípios"]
    resumo = pd.DataFrame(index=rotulos)

    for indicador in [coluna_credito, coluna_producao]:
        valores = dados[indicador].sort_values(ascending=False)
        total = valores.sum()
        participacoes = []
        for percentual in faixas:
            # Arredondar para cima para incluir pelo menos um município.
            quantidade = (len(valores) * percentual + 99) // 100
            parcela = valores.head(quantidade).sum()
            participacoes.append(parcela / total if total > 0 else float("nan"))
        # Somar o que ficou fora do Top 10% no ranking deste indicador.
        restante = valores.iloc[quantidade:].sum()
        participacoes.append(restante / total if total > 0 else float("nan"))
        resumo[indicador] = participacoes

    primeiro_ano, ultimo_ano = periodos_g3.loc[cultura]
    ax = (resumo * 100).plot.bar(
        ylabel="Participação no total nacional (%)",
        color=["#2866A6", "#3A851D"],
        figsize=(11, 6),
        rot=0,
        width=0.7,
        ylim=(0, 100),
    )
    ax.set_title(
        f"Concentração por município · {cultura} ({primeiro_ano} a {ultimo_ano})", pad=32,
    )
    ax.legend(loc="lower center", bbox_to_anchor=(0.5, 1.02), ncol=2, frameon=False)
    ax.get_figure().tight_layout()
    plt.show()
    plt.close(ax.get_figure())

seletor_g3 = Dropdown(
    options=periodos_g3.index.tolist(), description="Cultura", layout=Layout(width="330px"),
)
saida_g3 = interactive_output(grafico_3, {"cultura": seletor_g3})
display(seletor_g3, saida_g3)

Dropdown(description='Cultura', layout=Layout(width='330px'), options=('Milho (em grão)', 'Soja (em grão)'), v…

Output()

### Gráfico 6) Municípios com maior crédito por hectare apresentam distribuições diferentes de produtividade, na mesma cultura e ano?

Os dois mapas colorem cada município pelo quartil de crédito de custeio por hectare plantado (azul) e pelo quartil de produtividade (verde, rendimento médio em t/ha). Se o crédito por hectare estivesse ligado à produtividade, as cores escuras apareceriam nos mesmos lugares. O boxplot à direita resume a mesma pergunta: a distribuição da produtividade em cada faixa de crédito por hectare (números em negrito = medianas). O recorte (Brasil, Grande Região ou UF) define onde os quartis são calculados.

**Decisões do tratamento:**
- As malhas vêm do `geobr` (municípios e UFs, ano 2020), ligadas aos dados pelo código IBGE do município. A junção IBGE × SICOR também usa o código, e não o nome, para não misturar municípios homônimos.
- Municípios com menos de 50 ha plantados (`AREA_MINIMA_HA`) são excluídos, porque denominadores pequenos inflam o crédito por hectare.
- Municípios sem crédito de custeio ou sem rendimento aparecem como "Sem dados".

In [41]:
# Base comum aos gráficos 6 e 7: crédito de custeio, área e produtividade por município, cultura e ano.
# A junção usa o código IBGE do município (como no gráfico 3) e não o nome, porque há municípios homônimos em UFs diferentes.
AREA_MINIMA_HA = 50  # evita denominadores muito pequenos em crédito por hectare

credito_g67 = (
    sicor.assign(ano=sicor["ano"].astype(int), municipio_id=sicor["codIbge"].astype(str))
    .groupby(["cultura", "ano", "municipio_id"], as_index=False)["valor de custeio"].sum()
    .rename(columns={"valor de custeio": "credito"})
)
ibge_g67 = tabela_ibge_municipal.assign(
    municipio_id=tabela_ibge_municipal["codigo municipio"].astype(str)
).rename(columns={"area plantada (hectares)": "area"})

base_g67 = ibge_g67.merge(credito_g67, on=["cultura", "ano", "municipio_id"])
base_g67 = base_g67[
    (base_g67["area"] >= AREA_MINIMA_HA)
    & (base_g67["credito"] > 0)
    & (base_g67["rendimento medio (kg por hectare)"] > 0)
].copy()
base_g67["credito_ha"] = base_g67["credito"] / base_g67["area"]
base_g67["produtividade"] = base_g67["rendimento medio (kg por hectare)"] / 1000  # t/ha

# Malhas do IBGE via geobr (o geobr guarda o download em cache local).
# Compatibilidade com duckdb 2.x: versões novas retiraram Relation.fetchone().
if not hasattr(duckdb.DuckDBPyRelation, "fetchone"):
    def _duckdb_fetchone(relation):
        linhas = relation.limit(1).fetchall()
        return tuple(linhas[0]) if linhas else None

    duckdb.DuckDBPyRelation.fetchone = _duckdb_fetchone

mapa_municipios = geobr.read_municipality(code_muni="all", year=2020)
mapa_estados = geobr.read_state(code_state="all", year=2020)
mapa_municipios = mapa_municipios.assign(municipio_id=mapa_municipios["code_muni"].astype("int64").astype(str))

# Recortes do mapa: Brasil, cada Grande Região (1º dígito do código do município; dicionário `regioes` do gráfico 2) ou uma UF.
mapa_municipios["regiao"] = mapa_municipios["municipio_id"].str[0].map(regioes)
recortes_g67 = {"Brasil": sorted(mapa_municipios["abbrev_state"].unique())}
for nome_regiao in regioes.values():
    recortes_g67[nome_regiao] = sorted(
        mapa_municipios.loc[mapa_municipios["regiao"] == nome_regiao, "abbrev_state"].unique()
    )
for uf in recortes_g67["Brasil"]:
    recortes_g67[uf] = [uf]


def desenhar_fundo(eixo, ufs):
    """Todos os municípios do recorte em cinza-claro e o contorno das UFs."""
    mapa_municipios[mapa_municipios["abbrev_state"].isin(ufs)].plot(ax=eixo, color="#E9EBED", linewidth=0)
    mapa_estados[mapa_estados["abbrev_state"].isin(ufs)].boundary.plot(ax=eixo, color="#555555", linewidth=0.6)
    eixo.set_axis_off()


faixas_g6 = ["Q1\n(menor)", "Q2", "Q3", "Q4\n(maior)"]
cores_credito = ["#D6E4F2", "#9BC1E3", "#5B97CF", "#2866A6"]
cores_produtividade = ["#D5E8CF", "#A8D19A", "#6FB05B", "#3A851D"]


def grafico_6(cultura, ano, recorte):
    ufs = recortes_g67[recorte]
    dados = base_g67[
        (base_g67["cultura"] == cultura) & (base_g67["ano"] == ano) & base_g67["estado"].isin(ufs)
    ].copy()
    if len(dados) < 8:
        print("Poucos municípios com crédito e produtividade para esta combinação.")
        return

    # Quartis dentro do recorte; o rank desempata valores iguais.
    dados["faixa_credito"] = pd.qcut(dados["credito_ha"].rank(method="first"), 4, labels=faixas_g6)
    dados["faixa_produtividade"] = pd.qcut(dados["produtividade"].rank(method="first"), 4, labels=faixas_g6)
    dados_mapa = mapa_municipios[["municipio_id", "geometry"]].merge(dados, on="municipio_id")

    fig, (ax_credito, ax_produtividade, ax_caixas) = plt.subplots(
        1, 3, figsize=(21, 7.5), layout="constrained", gridspec_kw={"width_ratios": [1, 1, 1.05]},
    )

    # Dois mapas lado a lado: se as cores se repetirem nos mesmos lugares, crédito e produtividade andam juntos.
    for eixo, coluna, cores, titulo in [
        (ax_credito, "faixa_credito", cores_credito, "Crédito de custeio por hectare"),
        (ax_produtividade, "faixa_produtividade", cores_produtividade, "Produtividade (t/ha)"),
    ]:
        desenhar_fundo(eixo, ufs)
        for faixa, cor in zip(faixas_g6, cores):
            dados_mapa[dados_mapa[coluna] == faixa].plot(ax=eixo, color=cor, linewidth=0)
            eixo.scatter([], [], marker="s", s=90, color=cor, label=faixa.replace("\n", " "))
        eixo.scatter([], [], marker="s", s=90, color="#E9EBED", label="Sem dados")
        eixo.legend(loc="lower left", frameon=False, title="Quartil no recorte", fontsize=10)
        eixo.set_title(titulo, fontsize=14)

    # Boxplot: a produtividade em cada faixa de crédito por hectare.
    grupos = [dados.loc[dados["faixa_credito"] == f, "produtividade"] for f in faixas_g6]
    caixas = ax_caixas.boxplot(
        grupos, patch_artist=True, showfliers=False, widths=0.6,
        medianprops={"color": "black", "linewidth": 2},
    )
    for caixa, cor in zip(caixas["boxes"], cores_credito):
        caixa.set(facecolor=cor, edgecolor="#444444")
    for posicao, grupo in enumerate(grupos, start=1):
        ax_caixas.text(
            posicao, grupo.median(), f" {grupo.median():.1f}",
            ha="left", va="bottom", fontsize=11, fontweight="bold",
        )
    ax_caixas.set_xticks(range(1, 5))
    ax_caixas.set_xticklabels([f"{f}\nn = {len(g)}" for f, g in zip(faixas_g6, grupos)], fontsize=11)
    ax_caixas.set(
        xlabel="Faixa de crédito de custeio por hectare",
        ylabel="Produtividade (t/ha)",
    )
    ax_caixas.set_title("Produtividade em cada faixa de crédito", fontsize=14)
    ax_caixas.set_axisbelow(True)
    ax_caixas.grid(axis="y", alpha=0.22)
    ax_caixas.spines[["top", "right"]].set_visible(False)

    fig.suptitle(f"Crédito por hectare e produtividade · {cultura} · {ano} · {recorte}", fontsize=18)
    plt.show()
    plt.close(fig)


seletor_cultura_g6 = Dropdown(
    options=sorted(base_g67["cultura"].unique()), description="Cultura", layout=Layout(width="330px"),
)
anos_g6 = sorted(base_g67["ano"].unique())
seletor_ano_g6 = Dropdown(
    options=anos_g6, value=anos_g6[-1], description="Ano", layout=Layout(width="330px"),
)
seletor_recorte_g6 = Dropdown(
    options=list(recortes_g67), description="Recorte", layout=Layout(width="330px"),
)
saida_g6 = interactive_output(
    grafico_6, {"cultura": seletor_cultura_g6, "ano": seletor_ano_g6, "recorte": seletor_recorte_g6},
)
display(seletor_cultura_g6, seletor_ano_g6, seletor_recorte_g6, saida_g6)

states_2020_simplified.parquet: 100%|██████████| 1.72M/1.72M [00:00<00:00, 2.26MB/s]


Dropdown(description='Cultura', layout=Layout(width='330px'), options=('Milho (em grão)', 'Soja (em grão)'), v…

Dropdown(description='Ano', index=9, layout=Layout(width='330px'), options=(np.int64(2016), np.int64(2017), np…

Dropdown(description='Recorte', layout=Layout(width='330px'), options=('Brasil', 'Norte', 'Nordeste', 'Sudeste…

Output()

### Gráfico 7) Quais municípios apresentam valores atípicos de crédito relativo ou produtividade no padrão dos estados selecionados?

O mapa mostra onde estão os municípios atípicos: azul para crédito por hectare atípico, verde para produtividade atípica e vermelho para ambos. Um município é atípico quando está fora de [Q1 − 1,5·IQR, Q3 + 1,5·IQR] **dentro da própria UF** (crédito por hectare avaliado em log10). Na escolha de uma UF, os seis casos mais extremos são nomeados no mapa; a tabela abaixo lista os 15 mais extremos do recorte.

**Decisões do tratamento:**
- O padrão é sempre o da UF inteira; o recorte (Brasil, Grande Região ou UF) só decide o que é exibido.
- Mesma base do gráfico 6 (código IBGE, mínimo de 50 ha plantados).
- UFs com menos de 10 municípios na cultura e ano escolhidos não entram, pois não permitem estimar um padrão estável.
- O escore da tabela é o desvio em relação à mediana da UF, em unidades de desvio robusto (IQR/1,349).
- O dicionário `regioes` vem da célula do gráfico 2.

In [42]:
MIN_MUNICIPIOS_UF = 10  # UFs com poucos municípios não permitem estimar um padrão estável
cores_g7 = {
    "Crédito/ha atípico": "#2866A6",
    "Produtividade atípica": "#3A851D",
    "Ambos": "#C0392B",
}


def sinalizar_atipicos(dados):
    """Marca valores fora de [Q1 - 1,5·IQR, Q3 + 1,5·IQR] dentro de cada UF.

    O crédito por hectare é avaliado em log10, por ser muito assimétrico.
    O escore mede o desvio em relação à mediana da UF, em unidades de desvio robusto (IQR/1,349).
    """
    dados = dados.copy()
    dados["log_credito_ha"] = dados.eval("log10(credito_ha)")
    for coluna, nome in [("log_credito_ha", "credito"), ("produtividade", "produtividade")]:
        grupo = dados.groupby("estado")[coluna]
        q1, q3 = grupo.transform(lambda s: s.quantile(0.25)), grupo.transform(lambda s: s.quantile(0.75))
        iqr = q3 - q1
        mediana = grupo.transform("median")
        dados[f"atipico_{nome}"] = ((dados[coluna] < q1 - 1.5 * iqr) | (dados[coluna] > q3 + 1.5 * iqr)) & (iqr > 0)
        dados[f"escore_{nome}"] = ((dados[coluna] - mediana) / (iqr / 1.349).replace(0, float("nan"))).abs()

    ambos = dados["atipico_credito"] & dados["atipico_produtividade"]
    dados["tipo"] = (
        pd.Series("Típico", index=dados.index)
        .mask(dados["atipico_credito"], "Crédito/ha atípico")
        .mask(dados["atipico_produtividade"], "Produtividade atípica")
        .mask(ambos, "Ambos")
    )
    dados["escore"] = dados[["escore_credito", "escore_produtividade"]].max(axis=1)
    return dados


def grafico_7(cultura, ano, recorte):
    ufs = recortes_g67[recorte]
    # O padrão é o de cada UF inteira; o recorte só decide o que aparece no mapa.
    dados = base_g67[(base_g67["cultura"] == cultura) & (base_g67["ano"] == ano)]
    dados = dados[dados["estado"].map(dados["estado"].value_counts()) >= MIN_MUNICIPIOS_UF]
    if dados.empty:
        print("Nenhuma UF com municípios suficientes para esta cultura e ano.")
        return
    dados = sinalizar_atipicos(dados)
    dados = dados[dados["estado"].isin(ufs)]
    if dados.empty:
        print(f"Nenhuma UF de '{recorte}' tem pelo menos {MIN_MUNICIPIOS_UF} municípios nesta cultura e ano.")
        return
    dados_mapa = mapa_municipios[["municipio_id", "geometry"]].merge(dados, on="municipio_id")

    fig, ax = plt.subplots(figsize=(11, 10), layout="constrained")
    desenhar_fundo(ax, ufs)
    dados_mapa[dados_mapa["tipo"] == "Típico"].plot(ax=ax, color="#B8BEC4", linewidth=0)
    ax.scatter([], [], marker="s", s=90, color="#B8BEC4", label=f"Típico ({(dados['tipo'] == 'Típico').sum()})")
    ax.scatter([], [], marker="s", s=90, color="#E9EBED", label="Sem dados / excluído")

    # Atípicos: polígono colorido + marcador no centro, para aparecerem mesmo na escala do Brasil.
    for tipo, cor in cores_g7.items():
        poligonos = dados_mapa[dados_mapa["tipo"] == tipo]
        if not poligonos.empty:
            poligonos.plot(ax=ax, color=cor, linewidth=0)
    atipicos = dados_mapa[dados_mapa["tipo"] != "Típico"].copy()
    atipicos["geometry"] = atipicos.geometry.representative_point()
    tamanho = 110 if len(ufs) == 1 else 45 if len(ufs) <= 9 else 22
    for tipo, cor in cores_g7.items():
        parte = atipicos[atipicos["tipo"] == tipo]
        ax.scatter(
            parte.geometry.x, parte.geometry.y, s=tamanho, color=cor, edgecolor="black",
            linewidth=0.6, zorder=3, label=f"{tipo} ({len(parte)})",
        )
    if len(ufs) == 1:  # só nomeia no mapa quando há espaço
        for _, linha in atipicos.nlargest(6, "escore").iterrows():
            ax.annotate(
                linha["municipio"], (linha.geometry.x, linha.geometry.y),
                xytext=(7, 7), textcoords="offset points", fontsize=9, zorder=4,
            )
    ax.legend(loc="lower left", frameon=False, fontsize=11, title="Dentro da própria UF")
    ax.set_title(f"Municípios atípicos em crédito por hectare e produtividade\n{cultura} · {ano} · {recorte}", fontsize=15)
    plt.show()
    plt.close(fig)

    ranking = dados[dados["tipo"] != "Típico"].nlargest(15, "escore")
    ranking = ranking[["estado", "municipio", "credito_ha", "produtividade", "tipo"]].rename(columns={
        "estado": "UF", "municipio": "Município", "credito_ha": "Crédito (R$/ha)",
        "produtividade": "Produtividade (t/ha)", "tipo": "Tipo",
    })
    display(ranking.round({"Crédito (R$/ha)": 0, "Produtividade (t/ha)": 1}).reset_index(drop=True))


seletor_cultura_g7 = Dropdown(
    options=sorted(base_g67["cultura"].unique()), description="Cultura", layout=Layout(width="330px"),
)
seletor_ano_g7 = Dropdown(
    options=anos_g6, value=anos_g6[-1], description="Ano", layout=Layout(width="330px"),
)
seletor_recorte_g7 = Dropdown(
    options=list(recortes_g67), description="Recorte", layout=Layout(width="330px"),
)
saida_g7 = interactive_output(
    grafico_7, {"cultura": seletor_cultura_g7, "ano": seletor_ano_g7, "recorte": seletor_recorte_g7},
)
display(seletor_cultura_g7, seletor_ano_g7, seletor_recorte_g7, saida_g7)

Dropdown(description='Cultura', layout=Layout(width='330px'), options=('Milho (em grão)', 'Soja (em grão)'), v…

Dropdown(description='Ano', index=9, layout=Layout(width='330px'), options=(np.int64(2016), np.int64(2017), np…

Dropdown(description='Recorte', layout=Layout(width='330px'), options=('Brasil', 'Norte', 'Nordeste', 'Sudeste…

Output()